# 3. Create Gold Table



#### not_purchased_items_silverからclicked_items_per_monthを作成

- クリックされたが購入に至らなかった商品のカウントを記録する

In [0]:
CREATE OR REPLACE TABLE no_purchase_counts_gold AS
SELECT t.product_id,
  count(*) AS no_purchase_count,
  current_timestamp() as ingestion_timestamp
FROM (
  SELECT *,
    explode(not_purchased_items) AS product_id
  FROM not_purchased_items_silver
) t
JOIN products_silver
  ON products_silver.product_id = t.product_id
GROUP BY t.product_id;

SELECT * FROM no_purchase_counts_gold ORDER BY no_purchase_count DESC;


#### order_products_silverからordered_product_counts_monthly_goldを作成

- 商品の月毎の購入回数を記録するテーブル

In [0]:
-- 購入回数集計（月別）
CREATE OR REPLACE TEMP VIEW tmp_purchase_counts_monthly AS
SELECT product_id,
  date_format(ordered_at, 'yyyy-MM') AS month,
  count(*) AS purchase_count,
  current_timestamp() as ingestion_timestamp
FROM order_products_silver
WHERE ordered_at IS NOT NULL
GROUP BY product_id, month;

CREATE OR REPLACE TABLE ordered_products_monthly_gold AS
SELECT 
  s.product_id,
  COALESCE(p.purchase_count, 0) AS purchase_count,
  p.month as ordered_at,
  current_timestamp() as ingestion_timestamp
FROM tmp_purchase_counts_monthly p 
LEFT JOIN products_silver s
  ON s.product_id = p.product_id;

SELECT * FROM ordered_products_monthly_gold ORDER BY purchase_count DESC;


#### clicked_items_silverからclicked_products_monthly_goldを作成

- 商品の月毎のクリック回数を記録するテーブル

In [0]:
-- 月毎の商品クリック回数
CREATE OR REPLACE TABLE clicked_products_monthly_gold AS
SELECT
    t.product_id,
    count(*) AS click_count,
    date_format(ordered_at, "yyyy-MM") AS month,
    current_timestamp() as ingestion_timestamp
FROM (
  SELECT *,
    explode(clicked_items) AS product_id
  FROM clicked_items_silver
) t
JOIN products_silver
  ON products_silver.product_id = t.product_id
GROUP BY t.product_id, month;


SELECT * FROM clicked_products_monthly_gold;

--------------------------------
------------

以下メモ

In [0]:
SELECT sum(purchase_count)
FROM cli

In [0]:
SELECT sum(purchase_count)
FROM ordered_product_counts_monthly_gold

In [0]:
SELECT sum(purchase_count)
FROM purchase_counts_gold;

In [0]:
SELECT SUM(click_count)
FROM clicked_items_per_month_gold

In [0]:

CREATE OR REPLACE TEMP VIEW tmp_purchase_counts
AS
SELECT product_id,
  ordered_at AS ordered_at,
  count(*) AS purchase_count,
  current_timestamp() as ingestion_timestamp
FROM order_products_silver
WHERE ordered_at IS NOT NULL
GROUP BY product_id, ordered_at;


-- 購入回数集計（月別）
CREATE OR REPLACE TEMP VIEW tmp_purchase_counts_monthly AS
SELECT 
  product_id,
  date_format(ordered_at, 'yyyy-MM') AS month,
  COUNT(*) AS purchase_count
FROM tmp_purchase_counts
WHERE ordered_at IS NOT NULL
GROUP BY product_id, month;

-- クリック回数集計
CREATE OR REPLACE TEMP VIEW tmp_click_counts AS
SELECT count(*) AS click_count,
    product_id,
    date_format(ordered_at, "yyyy-MM") AS month
FROM (
  SELECT *,
    explode(clicked_items) AS product_id
  FROM clicked_items_silver
)
GROUP BY product_id, month;

-- 月毎のクリック回数と購入回数
CREATE OR REPLACE TABLE ordered_product_counts_monthly_gold AS
SELECT 
  s.product_id,
  -- COALESCE(c.click_count, 0) AS click_count,
  COALESCE(p.purchase_count, 0) AS purchase_count,
  p.month as ordered_at,
  current_timestamp() as ingestion_timestamp
FROM products_silver s
LEFT JOIN tmp_click_counts c
  ON s.product_id = c.product_id
LEFT JOIN tmp_purchase_counts_monthly p
  ON s.product_id = p.product_id AND c.month = p.month;

In [0]:
SELECT *
FROM products_silver
JOIN sales_bronze
ON products_silver.product_id = sales_bronze.product:id
WHERE products_silver.product_name = sales_bronze.product_name



In [0]:
SELECT COUNT(*) 
FROM products_silver